# Самостоятельная работа. Нелинейная регрессия

Цель работы — обучить полносвязную модель через `nn.Sequential` и оценить её ответы на отложенных данных.
Циклы обучения, валидации и тестирования нужно написать самостоятельно.

## Набор данных

Таблица `df` содержит 1500 искусственных наблюдений.
Столбцы `x1` и `x2` — числовые признаки.
Столбец `target` — измеренная величина с небольшим случайным шумом.
Модель должна предсказывать `target` по двум признакам.

## Импорт

In [1]:
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Данные

In [2]:
rng = np.random.default_rng(17)

df = pd.DataFrame({
    "x1": rng.uniform(-3, 3, 1500),
    "x2": rng.uniform(-2, 2, 1500),
})
df["target"] = (
    1.2 * df["x1"] ** 2
    + 0.7 * df["x1"] * df["x2"]
    + 2 * np.sin(1.5 * df["x2"])
    + rng.normal(0, 0.4, len(df))
)

## 1. Предпросмотр данных

Выведите несколько строк, размер таблицы, типы столбцов и число пропусков.
Покажите описательные статистики.
Постройте гистограмму `target` и диаграмму рассеяния `x1` и `x2`, где цвет точки зависит от `target`.
Подпишите оси и шкалу цвета.
Достаточно таблиц и графиков, письменный вывод не требуется.

## 2. Подготовка данных, Dataset и DataLoader

### 2.1. Разделение и масштабирование

Выделите признаки `x1`, `x2` и целевой столбец `target`.
Все дальнейшие данные берите только из `df`.
Разделите строки на `train`, `valid` и `test` в пропорции 60/20/20.
Зафиксируйте `random_state`.


Сохраните исходные координаты и индексы строк для итоговых графиков.
Масштабируйте признаки по статистикам `train`.
К `valid` и `test` примените те же параметры.
Целевой столбец не масштабируйте.

Подсказка. Сначала отделите 20% строк для теста.
Затем выделите четверть оставшихся строк для валидации.
У `StandardScaler` используйте `fit` только для обучающих признаков.

### 2.2. Собственный Dataset

Напишите класс с методами `__init__`, `__len__` и `__getitem__`.
В конструкторе сохраните готовые признаки и ответы как тензоры `float32`.
Метод `__getitem__` должен возвращать одну пару «признаки, ответ».
Создайте отдельные объекты для `train`, `valid` и `test`.

Подсказка. Для всей части признаки имеют форму `(N, 2)`, а ответы `(N, 1)`.
Для одного объекта формы будут `(2,)` и `(1,)`.
Не вычисляйте масштабирование внутри `__getitem__`.

### 2.3. Загрузчики

Создайте `DataLoader` для каждой части.
Размер пакета выберите сами.
У обучающего загрузчика включите перемешивание, у двух остальных отключите.
Получите один пакет и выведите формы его тензоров.

Подсказка. Для итоговой оценки на `train` удобно создать ещё один загрузчик без перемешивания.
Он будет использовать тот же обучающий `Dataset`.

## 3. Функция потерь

Выберите функцию потерь для предсказания одного вещественного числа.
Создайте объект `loss_function`.

Подсказка. Рассмотрите `nn.MSELoss` или `nn.L1Loss`.
Используйте среднее значение потери по пакету.
Предсказания и правильные ответы должны иметь одинаковую форму `(B, 1)`.

## 4. Архитектура модели

Создайте полносвязную сеть через `nn.Sequential`.
Подайте на вход два признака и получите одно число на выходе.
Число скрытых слоёв и их ширину выберите сами.
Между линейными слоями добавьте функции активации.
Выведите модель и форму её ответа на один пакет.

Подсказка. После последнего линейного слоя активация не нужна.
Начните с небольшой сети с одной или двумя скрытыми ступенями.
Признаки и параметры модели должны находиться на одном устройстве.

## 5. Оптимизатор

Выберите оптимизатор и задайте шаг обучения.
Передайте ему параметры своей модели.

Подсказка. Можно начать с Adam или SGD.
Сначала разместите модель на выбранном устройстве, затем создайте оптимизатор.
Для другой архитектуры потребуется новый оптимизатор.

## 6. Цикл обучения и валидации

Напишите цикл по эпохам и вложенный цикл по пакетам `train`.
Для каждого пакета очистите градиенты, вычислите предсказание и потерю, выполните обратный проход и шаг оптимизатора.
После каждой эпохи проведите отдельный проход по `valid` без обновления весов.

Сохраняйте среднюю потерю и RMSE на `train` и `valid` для каждой эпохи.
При уменьшении потери на `valid` копируйте состояние модели.
После обучения восстановите лучшие сохранённые параметры.
Архитектуру и настройки выбирайте по `valid`, не используя `test`.

Подсказка. Перед обучающим проходом вызывайте `train()`.
Для валидации используйте `eval()` вместе с `torch.no_grad()`.
Среднюю потерю пакета умножайте на число его объектов, а итоговую сумму делите на размер выборки.
Для RMSE сложите квадраты ошибок всех объектов, разделите сумму на число объектов и извлеките корень.
Не усредняйте RMSE отдельных пакетов.
Копию весов удобно хранить через `deepcopy(model.state_dict())`.

## 7. Цикл тестирования

Напишите отдельный цикл по тестовому загрузчику.
Используйте модель с восстановленными параметрами.
Соберите все предсказания и правильные ответы, затем объедините каждый список в один тензор.
Сохраните среднюю тестовую потерю.

Подсказка. Здесь нужны `eval()` и `torch.no_grad()`.
Оптимизатор, `backward()` и обновление весов в этом цикле не используются.
Собирайте предсказания и ответы вместе, чтобы сохранить соответствие строк.

## 8. Метрики и история обучения

### 8.1. Итоговые метрики

Получите ответы сохранённой модели на `train` без обновления весов.
Рассчитайте MAE, RMSE и R² отдельно для `train` и `test`.
Соберите результаты в небольшую таблицу.

Подсказка. Для метрик нужны одномерные массивы одинаковой длины.
Уберите только последнюю ось у ответов формы `(N, 1)`.
Метрики сохранённой модели вычисляйте заново, а не берите из последней строки истории.

### 8.2. Графики по эпохам

Постройте график потерь на `train` и `valid`.
Отдельно постройте график RMSE на этих частях.
Отметьте эпоху, параметры которой сохранились.
Добавьте подписи осей и легенду.

Подсказка. Накопленные при обучении значения относятся к разным состояниям весов внутри эпохи.
Для сравнения метрик при одном состоянии модели можно делать дополнительный проход по `train` после эпохи.
Тестовые метрики по эпохам не вычисляйте.

## 9. Визуализация ответов модели

### 9.1. Правильные ответы и предсказания

Постройте диаграмму рассеяния для тестовых объектов.
По горизонтали отложите правильный `target`, по вертикали предсказание.
Добавьте линию точного совпадения и задайте одинаковый диапазон обеих осей.

### 9.2. Остатки

Вычислите остаток как правильный ответ минус предсказание.
Постройте зависимость остатка от предсказания на тестовой части.
Добавьте горизонтальную линию нулевой ошибки.

### 9.3. Линии уровня

Создайте равномерную сетку по `x1` и `x2` в пределах обучающих данных.
Получите предсказание модели для каждой точки сетки.
Постройте линии равного предсказанного значения через `contour` и добавьте подписи уровней.
Поверх нанесите тестовые объекты с цветом по их правильному `target`.
Используйте общую шкалу значений для точек и линий.

Подсказка. `meshgrid` задаёт две матрицы координат.
Для модели объедините их в таблицу с двумя столбцами.
Масштабируйте её тем же объектом, который подготовлен на `train`.
Вектор предсказаний верните к форме сетки через `reshape`.
Оси графика оставьте в исходных единицах.